# NOTEBOOK HUẤN LUYỆN FINE-TUNING EMBEDDING MODEL CHO MỸ PHẨM & SPA
### Khóa Luận Tốt Nghiệp: Hệ thống Chatbot Tư vấn E-Commerce & Liệu trình Làm đẹp
---
Notebook này cho phép bạn huấn luyện mô hình Embedding chuyên biệt trên **GPU T4 miễn phí** của Google Colab trong vòng **15 - 20 phút** mà không lo máy cá nhân bị yếu hay nóng máy.

**Các bước thực hiện:**
1. Cài đặt thư viện (`sentence-transformers`, `torch`).
2. Tải lên 2 file dữ liệu: `train_triplets.json` và `val_triplets.json`.
3. Chạy huấn luyện với hàm mất mát `MultipleNegativesRankingLoss`.
4. Đánh giá và tải file mô hình hoàn chỉnh về máy.

## Bước 1: Kiểm tra GPU và Cài đặt Thư viện

In [ ]:
!nvidia-smi
!pip install -q sentence-transformers torch numpy scikit-learn

## Bước 2: Tải lên dữ liệu huấn luyện
Bấm vào nút Upload bên dưới để tải lên 2 file:
- `train_triplets.json`
- `val_triplets.json`
*(Được sinh ra từ máy tính của bạn qua script `1_generate_dataset.py`)*

In [ ]:
from google.colab import files
import os

print("Hãy tải lên 2 file: train_triplets.json và val_triplets.json")
uploaded = files.upload()
print("Đã tải lên các file:", list(uploaded.keys()))

## Bước 3: Cấu hình và Bắt đầu Huấn luyện (Fine-tuning)

In [ ]:
import json
import math
import torch
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation

# 1. Đọc dữ liệu
with open("train_triplets.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open("val_triplets.json", "r", encoding="utf-8") as f:
    val_data = json.load(f)

print(f"Tập Train: {len(train_data)} mẫu | Tập Val: {len(val_data)} mẫu")

# 2. Chuẩn bị tập Train DataLoader
train_examples = []
for item in train_data:
    a = item.get("anchor", "").strip()
    p = item.get("positive", "").strip()
    n = item.get("negative", "").strip()
    if a and p:
        if n:
            train_examples.append(InputExample(texts=[a, p, n]))
        else:
            train_examples.append(InputExample(texts=[a, p]))

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

# 3. Chuẩn bị Bộ đánh giá (Evaluator)
val_anchors = [item["anchor"] for item in val_data if item.get("anchor") and item.get("positive") and item.get("negative")]
val_positives = [item["positive"] for item in val_data if item.get("anchor") and item.get("positive") and item.get("negative")]
val_negatives = [item["negative"] for item in val_data if item.get("anchor") and item.get("positive") and item.get("negative")]

evaluator = evaluation.TripletEvaluator(
    anchors=val_anchors,
    positives=val_positives,
    negatives=val_negatives,
    name="cosmetics_colab_val",
    show_progress_bar=True
)

# 4. Tải Base Model
BASE_MODEL_NAME = "bkai-foundation-models/vietnamese-bi-encoder"
print(f"Đang tải mô hình nền: {BASE_MODEL_NAME}...")
model = SentenceTransformer(BASE_MODEL_NAME)

# Đo điểm số trước khi train
print("\n--- ĐO ĐỘ CHÍNH XÁC BAN ĐẦU (BEFORE) ---")
score_before = evaluator(model)
print(f"Độ chính xác trước khi train: {score_before * 100:.2f}%")

# 5. Huấn luyện với MultipleNegativesRankingLoss
train_loss = losses.MultipleNegativesRankingLoss(model)
EPOCHS = 3
OUTPUT_PATH = "./my_cosmetics_embedding"

print("\n--- BẮT ĐẦU HUẤN LUYỆN TRÊN GPU T4 ---")
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=EPOCHS,
    warmup_steps=math.ceil(len(train_dataloader) * EPOCHS * 0.1),
    output_path=OUTPUT_PATH,
    save_best_model=True,
    show_progress_bar=True
)

# 6. Đánh giá sau khi train
best_model = SentenceTransformer(OUTPUT_PATH)
score_after = evaluator(best_model)
print("\n" + "="*60)
print(f"ĐỘ CHÍNH XÁC SAU KHI TRAIN (AFTER): {score_after * 100:.2f}%")
print(f"MỨC ĐỘ CẢI THIỆN: +{(score_after - score_before) * 100:.2f}%")
print("="*60)

## Bước 4: Nén và Tải Mô hình về Máy tính

In [ ]:
import shutil

# Nén thư mục mô hình thành file zip
shutil.make_archive("my_cosmetics_embedding", "zip", "my_cosmetics_embedding")
print("Đã nén xong thành file my_cosmetics_embedding.zip!")

# Tự động tải về máy
files.download("my_cosmetics_embedding.zip")
print("Đang tải file mô hình về máy của bạn...")